# Joint optimization — figures

Every figure is drawn here; the `.py` scripts only produce data. Run them first:

```
python pilot.py                 # chooses the pins for exps 1-3
python exp1_spread.py
python exp2_exploration.py
python exp3_generalization.py
python exp4_ratios.py           # ~30 min
python paths.py
```

Vocabulary is in `experiments/CONTEXT.md`; the mechanics of the three axes are in ADR-0023.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots

import landscape as L
import sweep

DATA = sweep.DATA
SEQ = "viridis"


def load(name):
    return np.load(DATA / f"{name}.npz")


def surface_pair(z_speed, z_best, x, y, xlab, ylab, title, logaxes=True):
    """The two panels every experiment produces: convergence speed and best fitness.

    Height is the metric, colour is the across-seed std dev -- so a low, evenly-coloured region is
    a setting that is both good and reliable, and a low but hot region is a lottery.
    """
    fig = make_subplots(
        rows=1, cols=2, specs=[[{"type": "surface"}, {"type": "surface"}]],
        subplot_titles=("convergence speed (evals)", "best design fitness"),
        horizontal_spacing=0.06,
    )
    for c, (m, s) in enumerate([z_speed, z_best], start=1):
        fig.add_trace(
            go.Surface(x=x, y=y, z=m.T, surfacecolor=s.T, colorscale=SEQ,
                       colorbar=dict(title="seed std", x=0.44 if c == 1 else 1.02, len=0.8)),
            row=1, col=c,
        )
    axes = dict(type="log") if logaxes else {}
    for scene in ("scene", "scene2"):
        fig.update_layout(**{scene: dict(
            xaxis=dict(title=xlab, **axes), yaxis=dict(title=ylab, **axes),
            zaxis=dict(title=""), camera=dict(eye=dict(x=1.6, y=-1.6, z=1.1)))})
    fig.update_layout(title=title, height=560, margin=dict(l=0, r=0, t=80, b=0))
    return fig


def metrics(z, ax0, ax1):
    """(speed, best) each as (mean, std) over seeds, on the two swept axes."""
    curves = torch.as_tensor(z["curves"]).squeeze()          # (n0, n1, seeds, C)
    best = torch.as_tensor(z["best"]).squeeze()              # (n0, n1, seeds)
    n0, n1, s, _ = curves.shape
    speed = sweep.convergence_evals(curves.reshape(-1, curves.shape[-1]), z["evals"])
    speed = speed.reshape(n0, n1, s).float()
    pack = lambda t: (t.mean(-1).numpy(), t.std(-1).numpy())
    return pack(speed), pack(best), z[f"axis_{ax0}"], z[f"axis_{ax1}"]

## Experiment 1 — Spread

Read the speed panel knowing that the update steps toward the sample cloud, so wider spread
mechanically produces larger steps. That coupling is deliberate (ADR-0023): wide sampling buying
faster coarse progress *is* the trade-off being measured.

In [ ]:
sp, bt, x, y = metrics(load("exp1_spread"), "sig_d", "sig_c")
surface_pair(sp, bt, x, y, "designer spread", "controller spread", "Experiment 1 — Spread").show()

## Experiment 2 — Exploration

The `e = 1` edges are degenerate by construction: nothing ever climbs there, so the pinned
generalization values have no effect at all. That flat edge is a finding, not an artifact.

In [ ]:
sp, bt, x, y = metrics(load("exp2_exploration"), "e_d", "e_c")
surface_pair(sp, bt, x, y, "designer exploration", "controller exploration",
             "Experiment 2 — Exploration", logaxes=False).show()

## Experiment 3 — Generalization

This is where the matching-radii hypothesis lives. The controller's radius is measured in the joint
space, so a designer ranging beyond it gets its good designs played badly — if the hypothesis holds,
the off-diagonal cells are worse than the diagonal at equal total radius.

In [ ]:
sp, bt, x, y = metrics(load("exp3_generalization"), "g_d", "g_c")
surface_pair(sp, bt, x, y, "designer generalization", "controller generalization",
             "Experiment 3 — Generalization").show()

In [ ]:
# Is the diagonal actually better? Compare matched radii against mismatched ones of equal sum.
z = load("exp3_generalization")
best = torch.as_tensor(z["best"]).squeeze().mean(-1).numpy()
g = z["axis_g_d"]
lg = np.log10(g)
S = lg[:, None] + lg[None, :]                    # total radius (log)
D = np.abs(lg[:, None] - lg[None, :])            # mismatch
bins = np.digitize(S, np.quantile(S, np.linspace(0, 1, 7)[1:-1]))
rows = []
for b in np.unique(bins):
    m = bins == b
    matched, mismatched = best[m & (D <= 0.4)], best[m & (D >= 1.2)]
    if matched.size and mismatched.size:
        rows.append((f"{S[m].mean():+.2f}", matched.mean(), mismatched.mean(),
                     mismatched.mean() - matched.mean()))
print(f"{'total radius':>12} {'matched':>9} {'mismatch':>9} {'penalty':>9}")
for r in rows:
    print(f"{r[0]:>12} {r[1]:9.4f} {r[2]:9.4f} {r[3]:+9.4f}")
print("\npositive penalty = mismatched radii are worse at the same total radius")

## Path taken

The landscape as a surface, with the pair's joint trajectory `(mu_d, mu_a)` drawn on it. Red shading
encodes recency — whiter is earlier — so a single static frame still carries the ordering. Green
points are the evaluated samples, subsampled.

In [ ]:
z = load("paths")
n = 256
Zg = L.f(L.grid(n)).numpy()
ax = L.axis(n).numpy()
names = [str(s) for s in z["names"]]

fig = make_subplots(rows=2, cols=2, specs=[[{"type": "surface"}] * 2] * 2,
                    subplot_titles=names, vertical_spacing=0.06, horizontal_spacing=0.04)
for i, name in enumerate(names):
    r, c = divmod(i, 2)
    fig.add_trace(go.Surface(x=ax, y=ax, z=Zg.T, colorscale="Blues_r", showscale=False,
                             opacity=0.9), row=r + 1, col=c + 1)
    md_, ma = z["mu_d"][:, i], z["mu_a"][:, i]
    t = np.linspace(0, 1, len(md_))
    zt = L.f(torch.tensor(np.stack([md_, ma], -1), dtype=torch.float32)).numpy() + 0.02
    fig.add_trace(go.Scatter3d(x=md_, y=ma, z=zt, mode="markers",
                               marker=dict(size=2.5, color=t, colorscale="Reds", cmin=-0.35),
                               showlegend=False), row=r + 1, col=c + 1)
    d, a = z["d"][:, i].ravel(), z["a"][:, i].ravel()
    k = np.random.default_rng(0).choice(d.size, min(1500, d.size), replace=False)
    zs = L.f(torch.tensor(np.stack([d[k], a[k]], -1), dtype=torch.float32)).numpy() + 0.01
    fig.add_trace(go.Scatter3d(x=d[k], y=a[k], z=zs, mode="markers",
                               marker=dict(size=1.3, color="#2ecc40", opacity=0.45),
                               showlegend=False), row=r + 1, col=c + 1)

for s in ("scene", "scene2", "scene3", "scene4"):
    fig.update_layout(**{s: dict(xaxis_title="design", yaxis_title="action", zaxis_title="value",
                                 camera=dict(eye=dict(x=1.5, y=-1.5, z=1.2)))})
fig.update_layout(height=1000, margin=dict(l=0, r=0, t=60, b=0),
                  title="Path taken — red shaded by recency (white early), green = samples")
fig.show()

## Experiment 4 — Sampling ratios

Four artifacts, each answering one question. The evaluation budget is identical across ratios by
construction, so any difference is structural rather than bought with compute.

In [ ]:
z4 = load("exp4_ratios")
PARAMS = [str(p) for p in z4["param_order"]]
ratios = z4["ratios"]
best4 = z4["best"]          # (5,5,5,5,5,5, ratio, seed)
speed4 = z4["speed"]
axes4 = {p: z4[f"axis_{p}"] for p in PARAMS}
mean4 = best4.mean(-1)      # over seeds -> (..., ratio)
print("grid", best4.shape, " ratios", ratios)

### 1. Sensitivity — how much does each param move the metric, at each ratio?

In [ ]:
# marginalize over the other five params, then measure the range the swept one spans
sens = np.zeros((len(PARAMS), len(ratios)))
for i in range(len(PARAMS)):
    other = tuple(j for j in range(len(PARAMS)) if j != i)
    marg = mean4.mean(axis=other)               # (5, ratio)
    sens[i] = marg.max(0) - marg.min(0)

go.Figure(go.Heatmap(z=sens, x=[f"1:{r}" for r in ratios], y=PARAMS, colorscale=SEQ,
                     colorbar=dict(title="impact"))).update_layout(
    title="Impact of each parameter, by sampling ratio", height=420,
    xaxis_title="sampling ratio", yaxis_title="").show()

### 2. Where the optimum sits, as the ratio changes

The headline figure. Uses the centroid of the top-N configs rather than the raw argmin, which jumps
discontinuously under seed noise; the centroid honestly represents *the region of good settings*.

In [ ]:
TOP_N = 64
flat = mean4.reshape(-1, len(ratios))
opt = np.zeros((len(PARAMS), len(ratios)))
for r in range(len(ratios)):
    idx = np.unravel_index(np.argsort(flat[:, r])[:TOP_N], mean4.shape[:-1])
    for i, p in enumerate(PARAMS):
        v = axes4[p][idx[i]]
        lo, hi = axes4[p].min(), axes4[p].max()
        opt[i, r] = (np.mean(v) - lo) / (hi - lo)      # normalized to its own sweep range

fig = go.Figure()
for i, p in enumerate(PARAMS):
    fig.add_trace(go.Scatter(x=ratios, y=opt[i], mode="lines+markers", name=p))
fig.update_layout(title=f"Optimal setting vs sampling ratio (centroid of top {TOP_N})",
                  xaxis=dict(title="sampling ratio (controller updates per designer update)",
                             type="log", tickvals=ratios, ticktext=[f"1:{r}" for r in ratios]),
                  yaxis_title="normalized optimum", height=480).show()

### 3. Strongest interactions — only the few worth looking at

In [ ]:
# second-order interaction: how much the joint effect departs from the sum of the marginals
pairs = []
for i in range(len(PARAMS)):
    for j in range(i + 1, len(PARAMS)):
        other = tuple(k for k in range(len(PARAMS)) if k not in (i, j))
        joint = mean4.mean(axis=other).mean(-1)                     # (5,5)
        add = joint.mean(1)[:, None] + joint.mean(0)[None, :] - joint.mean()
        pairs.append((np.abs(joint - add).mean(), PARAMS[i], PARAMS[j], i, j))
pairs.sort(reverse=True)

fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{a} x {b}" for _, a, b, _, _ in pairs[:3]])
for c, (_, a, b, i, j) in enumerate(pairs[:3], start=1):
    other = tuple(k for k in range(len(PARAMS)) if k not in (i, j))
    joint = mean4.mean(axis=other).mean(-1)
    fig.add_trace(go.Heatmap(z=joint.T, x=axes4[a], y=axes4[b], colorscale=SEQ,
                             showscale=c == 3), row=1, col=c)
    fig.update_xaxes(title=a, row=1, col=c)
    fig.update_yaxes(title=b, row=1, col=c)
fig.update_layout(title="Three strongest parameter interactions", height=420).show()
for s, a, b, _, _ in pairs:
    print(f"{a:>6} x {b:<6}  {s:.4f}")

### 4. Named archetypes across ratios

In [ ]:
def nearest(p, v):
    return int(np.abs(axes4[p] - v).argmin())

import paths as P
fig = go.Figure()
for name, cfg in P.ARCHETYPES.items():
    idx = tuple(nearest(p, cfg[p]) for p in PARAMS)
    fig.add_trace(go.Scatter(x=ratios, y=mean4[idx], mode="lines+markers", name=name))
fig.update_layout(title="Archetypes across sampling ratios (nearest grid cell)",
                  xaxis=dict(title="sampling ratio", type="log", tickvals=ratios,
                             ticktext=[f"1:{r}" for r in ratios]),
                  yaxis_title="best design fitness", height=460).show()